In [2]:
import pandas as pd
from scipy.stats import chi2_contingency
from itertools import combinations
from statsmodels.stats.multitest import multipletests

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
# import after saving these in notebook 07

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # MAC
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # PC

df_augmin = pd.read_csv(root + "Cohort_for_stats_Augmin.csv")
df_USP28 = pd.read_csv(root + "Cohort_for_stats_USP28.csv")

Chi-square test of independence

Tests whether cell category distribution depends on time bin.

Null hypothesis: distribution of 0/1/2 is the same in all bins.

Alternative: distributions differ.

Used widely for contingency tables.

In [ ]:
def getstatistics(counts):
    # Because your data are counts in a contingency table, the most statistically appropriate and stable method for pairwise bin comparisons is actually:
    # Chi-square test on the 3×2 table (categories × bins)
    # This avoids the multinomial regression instability entirely.

    table = counts.pivot_table(
    index = "category",
    columns = "time_bin",
    values = "count",
    aggfunc = "sum",
    fill_value = 0
    )
    print("Global results")
    print(chi2_contingency(table))
    
    
    expanded = counts.loc[counts.index.repeat(counts["count"])].drop(columns = "count")


    
    def compare_bins(counts_df, bin1, bin2):

        subset = counts_df[counts_df["time_bin"].isin([bin1, bin2])]
    
        table = subset.pivot_table(
            index="category",
            columns="time_bin",
            values="count",
            aggfunc="sum",
            fill_value=0
        )

        chi2, p, dof, expected = chi2_contingency(table)

        return p

    bins = counts["time_bin"].unique()

    results = []

    for b1, b2 in combinations(bins, 2):
        p = compare_bins(counts, b1, b2)
        results.append((b1, b2, p))
    
    pairwise = pd.DataFrame(results, columns=["bin1","bin2","p"])
    
    
    
    pairwise["p_adj"] = multipletests(pairwise["p"], method="fdr_bh")[1] # this is the Multiple testing correction (Benjamini–Hochberg (FDR))

    print("Pairwise results")
    print(pairwise)    

In [ ]:
getstatistics(df_augmin)

In [ ]:
getstatistics(df_USP28)